In [2]:
# A2-1 setup + review sheets
import os
from pathlib import Path
import pandas as pd
from PIL import Image, ImageDraw

root = Path.cwd()
while not (root / "data" / "imgs_cleaned").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
CLEAN = Path("data/imgs_cleaned"); A1 = Path("logs/preprocess_a1")
OUT = Path("logs/preprocess_a2"); (OUT / "sheets").mkdir(parents=True, exist_ok=True)
META = Path("data/meta"); META.mkdir(parents=True, exist_ok=True)

m = pd.read_csv(A1 / "measurements.csv")
tags = m.set_index("file")["flags"].fillna("")
box = set(pd.read_csv(A1 / "black_boxes.csv")["file"])
bars = set(pd.read_csv(A1 / "letterbox.csv")["file"])
SRC = {"existing_dataset": "old", "new_dataset": "NEW", "new_dataset_a": "NEW-a",
       "new_dataset_b": "NEW-b", "original": "orig"}
NOISY = {"padded", "grayscale", "mirror_like"}          # known false positives from A1

def label(f):
    t = [x for x in str(tags.get(f, "")).split(",") if x and x not in NOISY]
    if f in box: t.append("BOX")
    if f in bars: t.append("BARS")
    return SRC.get(m.set_index("file").at[f, "source"], "?") + " " + ",".join(t)

def make_page(files, path, cols=8, rows=6, th=160):
    sheet = Image.new("RGB", (cols * th, rows * (th + 22)), "white"); dr = ImageDraw.Draw(sheet)
    for i, f in enumerate(files):
        im = Image.open(CLEAN / f).convert("RGB"); im.thumbnail((th, th))
        x, y = (i % cols) * th, (i // cols) * (th + 22)
        sheet.paste(im, (x, y))
        dr.text((x + 2, y + th + 1), f.split("/")[-1].replace(".jpg", ""), fill="black")
        dr.text((x + 2, y + th + 11), label(f)[:28], fill="red")
    sheet.save(path, quality=90)

for p in (OUT / "sheets").glob("*.jpg"): p.unlink()
CLASSES = sorted(m["cls"].unique())
for c in CLASSES:
    sub = m[m["cls"] == c].sort_values(["source", "file"])
    fs = sub["file"].tolist()
    pages = -(-len(fs) // 48)
    for i in range(0, len(fs), 48):
        make_page(fs[i:i + 48], OUT / "sheets" / f"{c}_p{i // 48 + 1:02d}.jpg")
    print(f"{c:15s} {len(fs):4d} images -> {pages} pages | " +
          ", ".join(f"{SRC.get(s, s)}={n}" for s, n in sub["source"].value_counts().items()))

dosa             181 images -> 4 pages | NEW=132, old=49
hoppers          156 images -> 4 pages | old=81, NEW=75
idli              86 images -> 2 pages | old=50, NEW=36
kottu             66 images -> 2 pages | NEW-b=35, old=28, NEW-a=3
milk_rice        143 images -> 3 pages | old=72, NEW=71
paratha          114 images -> 3 pages | old=113, NEW=1
pittu            249 images -> 6 pages | orig=249
rice             293 images -> 7 pages | orig=293
string_hoppers    83 images -> 2 pages | NEW=63, old=20


In [3]:
# A2-2 record decisions -> data/meta/review.csv
#   file | drop    | nonfood / illustration / mixed / other
#   file | relabel | new class folder name (e.g. paratha)
#   file | note    | free text, image stays as is (watermark, collage, same photo as X ...)
DECISIONS = """
dosa_0090 | drop | illustration
dosa_0159 | drop | other (product packaging)
dosa_0163 | drop | mixed (batter graphic)
dosa_0136 | drop | mixed (idli + dosa collage)
dosa_0105 | relabel | idli
idli_0052 | drop | mixed (breakfast spread collage)
idli_0083 | drop | other (rava poori, not idli)
pittu_0225 | drop | other (flatbread, no pittu)
pittu_0226 | drop | other (curry only)
pittu_0227 | drop | other (curry only)
rice_0097 | drop | illustration
rice_0083 | drop | mixed (several dishes, no clear rice)
rice_0085 | drop | mixed
rice_0108 | drop | mixed
rice_0117 | drop | mixed
kottu_0032 | note | checkerboard background
kottu_0046 | note | checkerboard background
paratha_0038 | note | checkerboard background
paratha_0049 | note | checkerboard background
paratha_0062 | note | checkerboard background
rice_0094 | note | checkerboard background
"""
by_name = {Path(f).stem: f for f in m["file"]}
rows_ = []
for line in DECISIONS.splitlines():
    line = line.strip()
    if not line or line.startswith("#"): continue
    parts = [p.strip() for p in line.split("|")]
    if len(parts) != 3: print("SKIPPED (need 3 parts):", line); continue
    name, action, detail = parts
    f = by_name.get(Path(name).stem)
    if f is None: print("UNKNOWN FILE:", name); continue
    if action not in {"drop", "relabel", "note"}: print("UNKNOWN ACTION:", line); continue
    if action == "relabel" and detail not in CLASSES: print("UNKNOWN CLASS:", line); continue
    rows_.append(dict(file=f, cls=f.split("/")[0], action=action, detail=detail))
rv = pd.DataFrame(rows_, columns=["file", "cls", "action", "detail"])
dup = rv[rv.duplicated("file", keep=False)]
if len(dup): print("WARNING: same file listed twice:", dup.file.unique().tolist())
rv.to_csv(META / "review.csv", index=False)
print(len(rv), "decisions saved ->", META / "review.csv")
if len(rv): print(rv.groupby(["cls", "action"]).size().to_string())

21 decisions saved -> data\meta\review.csv
cls      action 
dosa     drop       4
         relabel    1
idli     drop       2
kottu    note       2
paratha  note       3
pittu    drop       3
rice     drop       5
         note       1
